In [ ]:
import pandas as pd
import numpy as np
!pip install matplotlib

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [16]:

!pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\USER\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [20]:
data = pd.read_csv('profit_prediction_regression.csv')
data.head()


,Marketing Spend,Administration,Transport,Area,Profit
0,114523.61,136897.80,471784.10,Dhaka,192261.83
1,162597.70,151377.59,443898.53,Ctg,191792.06
2,153441.51,101145.55,407934.54,Rangpur,191050.39
3,144372.41,118671.85,383199.62,Dhaka,182901.99
4,142107.34,91391.77,366168.42,Rangpur,166187.94


In [21]:
# Check missing values
print("Missing Values:\n", data.isnull().sum())

# Data types
print("\nData Types:\n", data.dtypes)

# Statistical summary
print("\nSummary Statistics:\n", data.describe())

# Correlation matrix
plt.figure(figsize=(6,4))
plt.imshow(data.corr(), cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.title("Correlation Matrix")
plt.xticks(range(len(data.columns)), data.columns, rotation=90)
plt.yticks(range(len(data.columns)), data.columns)
plt.show()


Missing Values:
 Marketing Spend    0
Administration     0
Transport          0
Area               0
Profit             0
dtype: int64

Data Types:
 Marketing Spend    float64
Administration     float64
Transport          float64
Area                object
Profit             float64
dtype: object

Summary Statistics:
        Marketing Spend  Administration      Transport         Profit
count        50.000000       50.000000      50.000000      50.000000
mean      73721.615600   121344.639600  211025.097800  112012.639200
std       45902.256482    28017.802755  122290.310726   40306.180338
min           0.000000    51283.140000       0.000000   14681.400000
25%       39936.370000   103730.875000  129300.132500   90138.902500
50%       73051.080000   122699.795000  212716.240000  107978.190000
75%      101602.800000   144842.180000  299469.085000  139765.977500
max      165349.200000   182645.560000  471784.100000  192261.830000


ValueError: could not convert string to float: 'Dhaka'

<Figure size 600x400 with 0 Axes>

In [22]:
# Separate features & target
X = data.drop('Profit', axis=1)
y = data['Profit']

# One-Hot Encoding (Correct for Regression)
X = pd.get_dummies(X, columns=['Area'], drop_first=True)

X.head()


,Marketing Spend,Administration,Transport,Area_Dhaka,Area_Rangpur
0,114523.61,136897.80,471784.10,True,False
1,162597.70,151377.59,443898.53,False,False
2,153441.51,101145.55,407934.54,False,True
3,144372.41,118671.85,383199.62,True,False
4,142107.34,91391.77,366168.42,False,True


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [24]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred_lin = lin_model.predict(X_test)


In [25]:
mse = mean_squared_error(y_test, y_pred_lin)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_lin)
r2 = r2_score(y_test, y_pred_lin)

print("Linear Regression Performance:")
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)


Linear Regression Performance:
MSE: 130390721.14438465
RMSE: 11418.875651498473
MAE: 9232.727124516587
R²: 0.8389824679943252


In [26]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

y_pred_lasso = lasso.predict(X_test)

print("Lasso R²:", r2_score(y_test, y_pred_lasso))
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))

print("\nSelected Features (Non-zero coefficients):")
print(X.columns[lasso.coef_ != 0])


Lasso R²: 0.8389856506321641
Lasso RMSE: 11418.762799423119

Selected Features (Non-zero coefficients):
Index(['Marketing Spend', 'Administration', 'Transport', 'Area_Dhaka',
       'Area_Rangpur'],
      dtype='object')


In [27]:
ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

print("Ridge R²:", r2_score(y_test, y_pred_ridge))
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))


Ridge R²: 0.8390456074358494
Ridge RMSE: 11416.636603006189


In [28]:
results = pd.DataFrame({
    "Model": ["Linear", "Lasso", "Ridge"],
    "R2 Score": [
        r2_score(y_test, y_pred_lin),
        r2_score(y_test, y_pred_lasso),
        r2_score(y_test, y_pred_ridge)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_lin)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge))
    ]
})

results


,Model,R2 Score,RMSE
0,Linear,0.838982,11418.875651
1,Lasso,0.838986,11418.762799
2,Ridge,0.839046,11416.636603


In [29]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for train_idx, test_idx in kf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    lin_model.fit(X_tr, y_tr)
    pred = lin_model.predict(X_te)
    
    cv_scores.append(r2_score(y_te, pred))

print("Cross Validation R² Scores:", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))


Cross Validation R² Scores: [0.8389824679943246, 0.8312145361340547, 0.8510681568473165, 0.9506783086791654, 0.8494543671037522]
Mean CV R²: 0.8642795673517227


In [30]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

ridge_scaled = Ridge(alpha=0.1)
ridge_scaled.fit(X_train_s, y_train_s)

print("Scaled Ridge R²:",
      r2_score(y_test_s, ridge_scaled.predict(X_test_s)))


Scaled Ridge R²: 0.83870404696697
